## Part 3: Features, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns — NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

the datatset has been cleaned with 0 duplicated rows for (raceId, driverId)

our target 'podium' is binary int8 with values [0,1]. the overall podium rate is around 13.7% which matches the expected 3/field-size ratio.

missing values are present in the qualifying-related columns q1_s, q2_S, q3_s, qual_position.

this is what we want to target when choosing the start year since the f1 knockout started after 2006.

In [ ]:
base.head()

In [ ]:
base.dtypes

In [ ]:
# Shape of the cleaned base table
print("base shape:", base.shape)

# Grain: one row per (raceId, driverId)
dupes = base.duplicated(["raceId", "driverId"]).sum()
print("duplicated (raceId, driverId):", dupes)
assert dupes == 0, "grain broken"

# Target column present
assert "podium" in base.columns, "target 'podium' missing"
print("target dtype:", base["podium"].dtype)
print("target values:", base["podium"].unique())

# Target rate on the raw base
print("\nOverall podium rate:", base["podium"].mean().round(4))

In [ ]:
yearly = base.groupby("year")["podium"].agg(["mean", "count"])
print("Podium rate, last 10 years:")
print(yearly.tail(10))

print("\nPodium rate, first 10 years:")
print(yearly.head(10))

In [ ]:
miss = base.isna().mean().sort_values(ascending=False)
print("Top 15 columns by missing share:")
print(miss.head(15).round(4))

print("\nColumns with zero missing:")
print(miss[miss == 0].index.tolist())

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-199x),
so we need to choose the earliest year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table — needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1994-2010:")
print(q_cov.loc[1994:2010])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(cov.index, cov.values, marker="o", label="has_qualifying")
ax.axvline(2003, color="red", linestyle="--", label="START_YEAR = 2003")

ax.set_title("Qualifying coverage per season")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with qualifying data")
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

We chose 2003 since it defines the start of a stable inclusion of qualifying data. 
Despite q2 and q3 missing for some of the years after and including 2003, we can handle it with reached_q2, reached_q3, and qual_format rather than imputation. 

In [ ]:
START_YEAR = 2003
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])

In [ ]:
model["reached_q2"]  = model["q2_s"].notna().astype(int)
model["reached_q3"]  = model["q3_s"].notna().astype(int)
model["qual_format"] = (model["year"] >= 2006).astype(int)

model[["year", "q1_s", "q2_s", "q3_s", "reached_q2", "reached_q3", "qual_format"]]
#qual_format tells us if that year was before the knockout era where the q2 and q3 
#were getting recorded starting from 2006
#we're gonna use it to see the reason q2 and q3 are missing within the data

In [ ]:
print("model shape:", model.shape)

In [ ]:
print("model shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

### Section 3.4: Grid and qualifying features

The prediction cutoff is after qualifying, and before the race. This means all the following columns are safe for our predicitons.

**Grid features**:
- `grid` is the actual starting position after any penalties, (0 = no grid slot, handled by pit_lane_flag).
- `pit_lane_flag` is 1 if the driver started from the pit lane where `grid==0`, it is 0 otherwise.

**Qualifying features**
- `qual_position` is the driver's rank in the qualifying session
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)`).
- `gap_to_pole_pct` is the percentage of how much slower a driver was than the pole sitter
  in the same race (the fastest driver). Per-race normalised, so comparable across eras.

**Missingness.** Qualifying data does not exist for most of F1 history, so
we cut the ML table at 2003 (see 3.3). From 2006 onward, `q2_s` and `q3_s`
are also structurally missing for drivers eliminated in earlier rounds;
we handle this with `reached_q2`, `reached_q3`, and `qual_format` rather
than imputing.  

** All are set during or before grid formation. None come
from the race itself.**

In [ ]:
# Grid features
model["pit_lane_flag"] = model["pitlane_start"].astype(int)

print("Grid feature summary:")
print(model[["grid", "pit_lane_flag"]].describe())
print("\nPit-lane starts share:", model["pit_lane_flag"].mean().round(4))
print("Rows with grid == 0:", (model["grid"] == 0).sum())

In [ ]:
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

print(model[["grid", "pit_lane_flag", "qual_position",
             "best_qual_s", "gap_to_pole_pct"]].describe())

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
used = {"grid", "pit_lane_flag", "qual_position", "best_qual_s", "gap_to_pole_pct"}
assert not (used & FORBIDDEN), "post-race column used as feature"
print("grid + qualifying leakage check OK")

### Section 3.5: Pre-race standings ###

The standings tables (`driver_standings`, `constructor_standings`) record
cumulative points and rank **after** each race. Using them directly would
leak the current race's result into the features.

Fix: merge `races` to bring `year` and `round` into the standings, sort
by season+round, then `groupby(id).shift(1)` so each row holds the
standings **after the previous round** = standings going into this race.

**Leakage assertion.** Round 1 of every season must have NaN pre-race
standings (there is no previous race). The assertion fails loudly if
this is violated.

**Missingness.** ~5–7% of rows (round 1 of each season). We do NOT impute.

In [ ]:
# Bring year/round in — needed to sort standings chronologically
races = tables["races"][["raceId", "year", "round"]].copy()
print(races.shape)
races.head()

In [ ]:
ds = tables["driver_standings"].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before"]].copy()
ds_pre.head()

In [ ]:
# What year/round is raceId 833?
print(races[races["raceId"] == 833])

# Look at more rows to see the shift working
# (filter to one driver across a season)
sample_driver = ds_pre["driverId"].iloc[0]
print("\nSample driver:", sample_driver)
sample = ds_pre.merge(races, on="raceId").query("driverId == @sample_driver")
print(sample.sort_values(["year", "round"]).head(10))

In [ ]:
# Merge pre-race driver standings into the modeling table
model = model.merge(ds_pre, on=["raceId", "driverId"], how="left")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before"]].isna().mean())

# Assertion: first race of each season must have NaN pre-race standings
first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].isna().all(), "LEAK: standings before round 1"
assert model.loc[first_race, "pos_before"].isna().all(), "LEAK: standings before round 1"
print("driver standings leakage check OK")

In [ ]:
# --- Constructor standings: standings going INTO each race ---
cs = tables["constructor_standings"].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs_pre = cs[["raceId", "constructorId", "cons_pts_before", "cons_pos_before"]].copy()
cs_pre.head()

In [ ]:
print(races[races["raceId"].isin([765, 766])])

In [ ]:
# Merge into model
model = model.merge(cs_pre, on=["raceId", "constructorId"], how="left")

print("model shape:", model.shape)
print(model[["cons_pts_before", "cons_pos_before"]].isna().mean())

# Assertion: round 1 of each season must have NaN
assert model.loc[first_race, "cons_pts_before"].isna().all(), "LEAK: constructor standings"
assert model.loc[first_race, "cons_pos_before"].isna().all(), "LEAK: constructor standings"
print("constructor standings leakage check OK")

### 3.6 Rolling form ###

We build a 5-race rolling podium rate for each driver and each constructor:
"how often did this driver/team podium in the last 5 races?"

The rule is that rolling windows must use only past races. We enforce this
with `shift(1)` before `.rolling()`: row N never sees its own race's result.

For constructors, we first collapse to one row per team-race, because two
drivers share the same constructor. Naive `groupby("constructorId").shift(1)`
would leak the teammate's same-race result into the form value.

Leakage assertions:
- First career race per driver = NaN
- First race per constructor = NaN
- Both drivers of a team-race share the same constructor_form5 value

In [ ]:
# Clean slate: drop the old form columns if they exist
model = model.drop(columns=["driver_form5", "constructor_form5"], errors="ignore")
print("model columns now:", model.shape)

In [ ]:
# Compute form on ALL years so pre-2003 history is available
base_sorted = base.sort_values(["driverId", "year", "round"]).reset_index(drop=True)

base_sorted["driver_form5"] = (
    base_sorted.groupby("driverId")["podium"]
               .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model (2003+)
model = model.merge(
    base_sorted[["raceId", "driverId", "driver_form5"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "year", "round", "podium", "driver_form5"]].head(10)

In [ ]:
# First career race ever (from base) must be NaN — check by looking at base_sorted
first_driver_career = base_sorted.groupby("driverId").cumcount() == 0
assert base_sorted.loc[first_driver_career, "driver_form5"].isna().all(), "LEAK: driver form"

# Also check: within model, no NaN except genuinely-new drivers
print("driver_form5 NaN share in model:", model["driver_form5"].isna().mean().round(4))
print("driver form leakage check OK")

In [ ]:
# Collapse to one row per (constructor, race) on ALL years
base_cons = (
    base.groupby(["constructorId", "raceId", "year", "round"], as_index=False)["podium"]
        .mean()
)

base_cons = base_cons.sort_values(["constructorId", "year", "round"])
base_cons["constructor_form5"] = (
    base_cons.groupby("constructorId")["podium"]
             .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
)

# Merge onto model
model = model.merge(
    base_cons[["raceId", "constructorId", "constructor_form5"]],
    on=["raceId", "constructorId"],
    how="left"
)

model[["constructorId", "year", "round", "podium", "constructor_form5"]].head(10)

In [ ]:
# 1. First race per constructor (chronologically) must be NaN
first_per_constructor = (
    base_cons.sort_values(["constructorId", "year", "round"])
            .groupby("constructorId")
            .head(1)
)
assert first_per_constructor["constructor_form5"].isna().all(), \
    "LEAK: constructor form on first race"

# 2. Both drivers of a team-race must share the same value
grouped = model.groupby(["raceId", "constructorId"])["constructor_form5"].nunique(dropna=False)
assert (grouped <= 1).all(), \
    "constructor form differs within the same team-race"

print("constructor form leakage checks OK")
print("constructor_form5 NaN share in model:", model["constructor_form5"].isna().mean().round(4))

In [ ]:
# Put rows back in chronological race order for downstream sections
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("model shape:", model.shape)
model[["raceId", "driverId", "year", "round",
       "driver_form5", "constructor_form5", "podium"]].head(5)

### Section 3.7: Circuit history + home flag + era ###

we have 3 feature groups to add. 

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We capture this with two features per
(driver, circuit):
- `circuit_starts_before` — number of previous races by this driver at this circuit
- `circuit_podiums_before` — number of previous podiums by this driver at this circuit

Both use only past races. The current race is excluded by `shift(1).cumsum()`.

We compute on the full `base` table (all years), then merge onto `model`
(2003+) so 2003 rows have their pre-2003 circuit history.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone).

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `pre_2014` (2003–2013), `hybrid` (2014–2021), `ground_effect` (2022+).
Year is known before the race.

In [ ]:
base_sorted_circuit = base.sort_values(["driverId", "circuitId", "year", "round"]).reset_index(drop=True)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)

base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Re-merge onto model (drop old columns first to avoid suffix mess)
model = model.drop(columns=["circuit_starts_before", "circuit_podiums_before"], errors="ignore")
model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before", "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left"
)

model[["driverId", "circuitId", "year", "round",
       "circuit_starts_before", "circuit_podiums_before", "podium"]].head(10)

In [ ]:
# Assertion on base_sorted_circuit (all years) — not on model
first_at_circuit_base = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_starts_before"] == 0).all(), \
    "LEAK: starts before"
assert (base_sorted_circuit.loc[first_at_circuit_base, "circuit_podiums_before"] == 0).all(), \
    "LEAK: podiums before"
print("circuit history leakage check OK")

In [ ]:
first = base_sorted_circuit.loc[first_at_circuit_base]
print("Total first-occurrence rows:", len(first))
print("Non-zero podiums:", (first["circuit_podiums_before"] != 0).sum())
print("NaN podiums:", first["circuit_podiums_before"].isna().sum())
print("\nSample:")
print(first[["driverId", "circuitId", "year", "circuit_podiums_before"]].head(10))

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina", "Australian": "Australia",
    "Austrian": "Austria", "Belgian": "Belgium", "Brazilian": "Brazil", "British": "UK",
    "Canadian": "Canada", "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France", "German": "Germany",
    "Hungarian": "Hungary", "Indian": "India", "Indonesian": "Indonesia", "Irish": "Ireland",
    "Italian": "Italy", "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland", "Portuguese": "Portugal",
    "Rhodesian": "Rhodesia", "Russian": "Russia", "South African": "South Africa",
    "Spanish": "Spain", "Swedish": "Sweden", "Swiss": "Switzerland", "Thai": "Thailand",
    "Uruguayan": "Uruguay", "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

# --- Drivers: nationality, stripped ---
drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

# --- Circuits: country, normalized ---
circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace("United States", "USA")

# --- Merge onto model ---
model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

# --- home_flag: circuit country matches driver's primary OR secondary home country ---
model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)

model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

assert model["nationality"].notna().all(), "driver missing nationality"
assert model["country"].notna().all(), "circuit missing country"
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"nationalities missing from mapping: {unmapped}"

print("home_flag value counts:")
print(model["home_flag"].value_counts())
print("\nHome starts per country (top 10):")
print(model[model["home_flag"] == 1]["country"].value_counts().head(10))

In [ ]:
# Keep only home_flag — drop helper columns from the join
model = model.drop(columns=["nationality", "country", "home_country", "second_home_country"],
                   errors="ignore")
print("model shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

In [ ]:
import pandas as pd
model["era"] = pd.cut(
    model["year"],
    bins=[2002, 2013, 2021, 2024],
    labels=["v8", "hybrid", "ground_effect"]
)
print(model["era"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary

The modeling table has the columns: 16 (keys + target + grid/qual raw), 
and 17 features grouped into 6 categories for the ablation, and `split` column.


**Note on identifiers.** `driverId`, `constructorId`, `circuitId`, and `number` 
are not in FEATURES — they are identifiers, not features. Their signal is 
captured by the historical aggregates (driver_form5, constructor_form5, 
circuit_starts_before, circuit_podiums_before).

**Note on pit_lane_flag.** This is the standardised int version of the raw 
`pitlane_start`. Only `pit_lane_flag` is in FEATURES; the raw column is not used.

In [ ]:
FEATURE_GROUPS = {
    "grid":       ["grid", "pit_lane_flag"],
    "qualifying": ["qual_position", "best_qual_s", "gap_to_pole_pct",
                   "reached_q2", "reached_q3", "qual_format"],
    "standings":  ["pts_before", "pos_before",
                   "cons_pts_before", "cons_pos_before"],
    "form":       ["driver_form5", "constructor_form5"],
    "circuit":    ["circuit_starts_before", "circuit_podiums_before"],
    "context":    ["home_flag"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

print(f"Total features: {len(FEATURES)}")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {len(cols)}")

**Note on `era`.** `era` was removed from FEATURES after the ablation
showed the context group adds ~0 to validation PR-AUC, and their
recommended dropping it. The ground-effect test era (2022+) never appears 
in training, so its one-hot encoding is uninformative. `era` stays in the 
table for EDA and reporting.

**Note on the `v8` label.** The `v8` era bucket spans 2003–2013, but F1 used 
V10 engines 2003–2005 and V8 engines 2006–2013. We label the whole window 
`v8` for simplicity; it's used for reporting, not as a model input.

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("all features are leakage-safe")

In [ ]:
import json
import os
os.makedirs("../reports", exist_ok=True)

with open("../reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

# Save as CSV instead of parquet
model.to_csv("../reports/p3_modeling_table.csv", index=False)

print("saved:")
print(" - reports/p3_feature_groups.json")
print(" - reports/p3_modeling_table.csv")
print(" - shape:", model.shape)

### Section 3.9: Evidence plots

We produce four plots to justify our features:

1. **Podium rate vs grid bucket** shows that starting position predicts podium.
2. **Podium rate vs qualifying bucket** same for qualifying position.
3. **Feature distributions by target** shows form features separate podium vs non-podium.
4. **Correlation heatmap** checks for redundancy and confirms features relate to `podium`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fig, ax = plt.subplots(figsize=(10, 5))

# Bucket grid into sensible groups
bins = [-1, 0, 2, 4, 6, 10, 15, 25]
labels = ["pit lane", "1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
model["grid_bucket"] = pd.cut(model["grid"], bins=bins, labels=labels)

grid_stats = model.groupby("grid_bucket", observed=True)["podium"].agg(["mean", "count"])
grid_stats["mean_pct"] = grid_stats["mean"] * 100

bars = ax.bar(grid_stats.index.astype(str), grid_stats["mean_pct"], color="tab:blue")
ax.bar_label(bars, fmt="%.1f%%", fontsize=9)
ax.set_title("Podium rate by grid position bucket")
ax.set_xlabel("Grid bucket")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 80)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_grid.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

bins_q = [0, 2, 4, 6, 10, 15, 25]
labels_q = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
model["qual_bucket"] = pd.cut(model["qual_position"], bins=bins_q, labels=labels_q)

qual_stats = model.groupby("qual_bucket", observed=True)["podium"].agg(["mean", "count"])
qual_stats["mean_pct"] = qual_stats["mean"] * 100

bars = ax.bar(qual_stats.index.astype(str), qual_stats["mean_pct"], color="tab:orange")
ax.bar_label(bars, fmt="%.1f%%", fontsize=9)
ax.set_title("Podium rate by qualifying position bucket")
ax.set_xlabel("Qualifying bucket")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 80)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_qual.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot form by target
axes[0].hist(model.loc[model["podium"] == 0, "driver_form5"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[0].hist(model.loc[model["podium"] == 1, "driver_form5"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[0].set_title("driver_form5 by target")
axes[0].set_xlabel("driver_form5")
axes[0].set_ylabel("Density")
axes[0].legend()

# Constructor form by target
axes[1].hist(model.loc[model["podium"] == 0, "constructor_form5"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[1].hist(model.loc[model["podium"] == 1, "constructor_form5"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[1].set_title("constructor_form5 by target")
axes[1].set_xlabel("constructor_form5")
axes[1].legend()

# Pre-race driver points by target
axes[2].hist(model.loc[model["podium"] == 0, "pts_before"].dropna(),
             bins=20, alpha=0.6, label="not podium", color="tab:red", density=True)
axes[2].hist(model.loc[model["podium"] == 1, "pts_before"].dropna(),
             bins=20, alpha=0.6, label="podium", color="tab:green", density=True)
axes[2].set_title("pts_before by target")
axes[2].set_xlabel("pts_before")
axes[2].legend()

plt.tight_layout()
plt.savefig("../reports/fig_p3_distributions_by_target.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# Numeric features only
numeric_features = [f for f in FEATURES if f != "era"]

corr = model[numeric_features + ["podium"]].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 8})
ax.set_title("Feature correlation heatmap")
plt.tight_layout()
plt.savefig("../reports/fig_p3_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

# Print correlations with podium, sorted
print("\nCorrelation with podium (sorted):")
print(corr["podium"].drop("podium").sort_values(ascending=False).round(3))

### 3.9 findings

**Plot 1 (grid):** podium rate drops monotonically from 72.5% (grid 1–2) 
to 1% (grid 16+). `grid` is a strong predictor.

**Plot 2 (qualifying):** same shape as grid. `qual_position` matters.

**Plot 3 (distributions):** `driver_form5`, `constructor_form5`, and 
`pts_before` all show clear class separation.

**Plot 4 (correlations with podium):**
- Strongest: `driver_form5` (0.55), `constructor_form5` (0.55), `qual_position` (-0.50), `grid` (-0.48)
- Moderate: `pts_before` (0.44), `circuit_podiums_before` (0.24)
- Weak: `home_flag` (-0.001), `qual_format` (-0.009), `pit_lane_flag` (-0.037)

**Notes:**
- `best_qual_s` (-0.055) is weak because absolute lap times vary by track; 
  `gap_to_pole_pct` (-0.28) is the normalized alternative and carries more signal.
- `home_flag` near-zero which is consistent with Q2 finding.
- No feature pair exceeds 0.95 correlation (see heatmap), so no severe redundancy.

### Section 3.10: Time split ###

We will split the 2003-2024 rows into: 
- **train**: year ≤ 2019
- **validation**: year 2020–2021
- **test**: year ≥ 2022

**Why these years.** 2020–21 gives us two full seasons (with one shortened 
by COVID) for hyperparameter tuning. 2022+ is a clean, larger test set 
(3 seasons) that includes the new ground-effect era. Test set is used once.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

print(model["split"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.to_csv("../reports/p3_modeling_table.csv", index=False)
print("saved with split — shape:", model.shape)
print("columns:", len(model.columns))

### Section 3.11: Preprocessing pipeline

We build a scikit-learn `ColumnTransformer`:

- **Numeric features**: `SimpleImputer(strategy="median")` → `StandardScaler()`
- **Categorical (`era`)**: `OneHotEncoder(handle_unknown="ignore")`

The pipeline is **fit on train only**, then applied to validation and test

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# All 17 features are numeric now
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, FEATURES),
])

print(f"preprocessor built for {len(FEATURES)} features")
print(FEATURES)

In [ ]:
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

X_train = train_df[FEATURES]
X_val   = val_df[FEATURES]
X_test  = test_df[FEATURES]

y_train = train_df["podium"].values
y_val   = val_df["podium"].values
y_test  = test_df["podium"].values

# Fit on TRAIN ONLY, then transform all three
preprocessor.fit(X_train)

X_train_t = preprocessor.transform(X_train)
X_val_t   = preprocessor.transform(X_val)
X_test_t  = preprocessor.transform(X_test)

print("Shapes:")
print(f"  X_train: {X_train.shape} -> {X_train_t.shape}")
print(f"  X_val:   {X_val.shape} -> {X_val_t.shape}")
print(f"  X_test:  {X_test.shape} -> {X_test_t.shape}")

In [ ]:
import numpy as np

# Sanity: no NaN after transform, values are roughly standardized
print("NaN after transform (should be 0):")
print(f"  X_train: {np.isnan(X_train_t).sum()}")
print(f"  X_val:   {np.isnan(X_val_t).sum()}")
print(f"  X_test:  {np.isnan(X_test_t).sum()}")

print("\nTrain mean ~0, std ~1 per feature:")
print(f"  mean range: {X_train_t.mean(axis=0).min():.4f} to {X_train_t.mean(axis=0).max():.4f}")
print(f"  std range:  {X_train_t.std(axis=0).min():.4f} to {X_train_t.std(axis=0).max():.4f}")

# Val/test means can be slightly off 0 (different distribution), but no NaN
print("\nVal mean range (may differ from 0):")
print(f"  {X_val_t.mean(axis=0).min():.4f} to {X_val_t.mean(axis=0).max():.4f}")

### 3.11 wrap-up

Pipeline: `SimpleImputer(strategy="median")` → `StandardScaler()`, fit on
the 6,781 training rows only, then applied to validation (774 rows) and
test (1,351 rows).

**Evidence of correctness:**
- No NaN after transform (imputer filled all missing values)
- Train mean ≈ 0, std ≈ 1 (scaler normalized correctly)
- Validation mean shifted to -0.43 to 0.43 (different year distribution, 
  as expected; the scaler was not fit on validation data, which would have 
  been leakage)

17 features, no one-hot encoding needed (era was dropped in 3.8).

### Section 3.12: Preprocessing experiments

Two experiments justify our preprocessing choices. Both use a small logistic 
regression only to measure the effect of the preprocessing.

**Experiment 1: scaling vs no scaling (validation)**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings |
|---|---|---|---|
| impute→scale | 0.7325 | 0.9298 | 0 |
| impute-only | 0.7321 | 0.9284 | 1 |

Metrics are nearly identical, but the unscaled version **fails to converge** 
(lbfgs hits the 2000-iteration limit). Scaling is required for the optimizer 
to work on this feature set, not just a nice-to-have.

**Experiment 2: fit-on-train vs fit-on-all (leakage demo)**
Demonstrates why we fit preprocessing on train only. The "fit on all" version 
sees validation/test data during fitting; its test score is optimistic, and 
the correct version is the honest estimate.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

# Pull splits
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

def lr_pipeline(scale: bool):
    """Logistic regression with median imputation; optionally scale."""
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("lr", LogisticRegression(max_iter=2000, random_state=SEED)))
    return Pipeline(steps)

def evaluate_pipeline(pipe, X_train, y_train, X_eval, y_eval, fit_on):
    """Fit pipe on fit_on data, evaluate on X_eval / y_eval."""
    pipe.fit(fit_on[0], fit_on[1])
    proba = pipe.predict_proba(X_eval)[:, 1]
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
    }

print("setup ready")

In [ ]:
import warnings
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

X_tr = train_df[FEATURES];  y_tr = train_df["podium"].values
X_va = val_df[FEATURES];    y_va = val_df["podium"].values

def run_and_count_warnings(pipe, X_fit, y_fit, X_eval, y_eval):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        pipe.fit(X_fit, y_fit)
        proba = pipe.predict_proba(X_eval)[:, 1]
    n_warn = sum(1 for x in w if issubclass(x.category, ConvergenceWarning))
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
        "convergence warnings": n_warn,
    }

rows = []
for name, scale in [("impute→scale", True), ("impute-only", False)]:
    r = run_and_count_warnings(lr_pipeline(scale=scale), X_tr, y_tr, X_va, y_va)
    rows.append({"pipeline": name, **r})

exp1 = pd.DataFrame(rows)
print("=" * 60)
print("Experiment 1 — scaling vs no scaling (validation)")
print("=" * 60)
print(exp1.to_string(index=False))

In [ ]:
# Experiment 1b — order: impute→scale vs scale→impute
pipe_imp_sc = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipe_sc_imp = Pipeline([
    ("sc",  StandardScaler()),
    ("imp", SimpleImputer(strategy="median")),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])

for name, pipe in [("impute→scale", pipe_imp_sc), ("scale→impute", pipe_sc_imp)]:
    pipe.fit(X_tr, y_tr)
    pr = average_precision_score(y_va, pipe.predict_proba(X_va)[:, 1])
    print(f"{name}: validation PR-AUC = {pr:.4f}")

In [ ]:
from sklearn.base import clone

# Isolate preprocessing leakage: classifier always trained on TRAIN labels,
# only the preprocessing (imputer + scaler) sees different data.
prep_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

def lr_on(prep, X, y):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(prep.transform(X), y)

# Correct: preprocessing fit on TRAIN features only
prep_correct = clone(prep_template).fit(X_tr)
clf_correct = lr_on(prep_correct, X_tr, y_tr)
proba_correct = clf_correct.predict_proba(prep_correct.transform(X_va))[:, 1]

# Leaky: preprocessing fit on TRAIN + VALIDATION features
# (no labels involved — only the median/mean/std statistics)
X_tr_va_features = pd.concat([X_tr, X_va])
prep_leaky = clone(prep_template).fit(X_tr_va_features)
clf_leaky = lr_on(prep_leaky, X_tr, y_tr)
proba_leaky = clf_leaky.predict_proba(prep_leaky.transform(X_va))[:, 1]

# Metrics
pr_correct = average_precision_score(y_va, proba_correct)
pr_leaky   = average_precision_score(y_va, proba_leaky)

print("Experiment 2 — preprocessing fit on train vs train+val (evaluated on VALIDATION)")
print(f"  Correct (preprocessing fit on train only) : PR-AUC = {pr_correct:.4f}")
print(f"  Leaky   (preprocessing fit on train+val)  : PR-AUC = {pr_leaky:.4f}")
print(f"  PR-AUC difference                         : {pr_leaky - pr_correct:+.4f}")

# Evidence that the leak is REAL, not seed noise:
# the scaler learned different statistics from the two fits.
scaler_correct = prep_correct.named_steps["scaler"]
scaler_leaky   = prep_leaky.named_steps["scaler"]

# Scaler drift expressed in standard-deviation units per feature
mean_shift_std = np.abs(scaler_correct.mean_ - scaler_leaky.mean_) / scaler_correct.scale_
scale_shift_pct = np.abs(scaler_correct.scale_ - scaler_leaky.scale_) / scaler_correct.scale_ * 100

print()
print("Scaler parameter drift (per-feature, std units):")
print(f"  Mean shift  : {mean_shift_std.mean():.4f} std (max {mean_shift_std.max():.4f})")
print(f"  Std change  : {scale_shift_pct.mean():.2f}%    (max {scale_shift_pct.max():.2f}%)")

### 3.12 findings

**Experiment 1: impute→scale vs impute-only (validation)**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings |
|---|---|---|---|
| impute→scale | 0.7325 | 0.9298 | 0 |
| impute-only  | 0.7321 | 0.9284 | 1 |

Metrics are nearly identical, but the **unscaled version fails to converge**
(lbfgs hits the 2000-iteration limit). Scaling is not a nicety — it's required
for the optimizer to work on this feature set. We use `SimpleImputer(strategy=
"median")` followed by `StandardScaler()` in the pipeline.

**Experiment 1b: does the order matter? (impute→scale vs scale→impute)**

Both orders converge and produce nearly the same validation PR-AUC 
(see the output above). We choose **impute→scale** because:
- It is the conventional order in scikit-learn pipelines.
- It produces a standardized final distribution (mean 0, std 1); the 
  scale→impute variant fills NaN with the median of scaled values, which 
  is not exactly 0.

The difference is expected to be small because our missingness is 
structural and small in volume (~7% of rows, mostly round-1 standings).

**Experiment 2: preprocessing fit on train vs train+val (leakage demo)**

The classifier is trained on train labels in both versions. Only the
preprocessing (imputer + scaler) sees different data during fitting.
Both are evaluated on **validation** — the test set is left untouched.

| Setup | Validation PR-AUC |
|---|---|
| Preprocessing fit on train only (correct) | 0.7325 |
| Preprocessing fit on train+val (leaky) | 0.7328 |
| Difference | **+0.0003** |

**Is the leak real, or is it noise?**

We compare the scaler's learned parameters between the two fits. Per
feature, the mean shifts by ~0.02 standard deviations on average
(max ~0.04), and the scale changes by a few percent on average, with a
max of ~27% on one feature (see output above). That max is on a feature
with high missingness — the round-1 rows contribute meaningfully to the
leaky fit. The leak is a real, measurable parameter drift.

**Why doesn't the metric move much?**

The PR-AUC impact is small (+0.0003): imputation and scaling are mild
transformations, so the leak's effect on a linear model's ranking is
small even though the scaler parameters moved.

**Why we still fit on train only:**

Even when the metric impact is small, the leak is a real violation: the
"leaky" pipeline's preprocessing statistics were influenced by data it
wasn't supposed to see. In a scenario with a larger distribution shift
(e.g. the ground-effect test era), the same violation could inflate the
score more. Fitting on train only is the honest, consistent rule.

**Takeaways:** StandardScaler is required (convergence), and fit-on-train-only
is the honest choice (leakage demo).

### Section 3.13: Feature-group ablation

We measure each feature group's contribution by dropping one group at a time, 
retraining the same model, and recording the drop in validation PR-AUC.

**Setup:** same model as the baseline (logistic regression), same 
preprocessing (impute median + scale), same train/val split. Only the feature 
set changes.

**Uses `train_eval()`** (from `src/f1_eval.py`) so the metrics are 
directly comparable to the baseline model's comparison table.

**Reading the results:** a large drop means the group is important. A near-zero 
or negative drop means the group adds nothing (or its signal is redundant with 
other groups).

In [ ]:
import sys
sys.path.append("../src")
from f1_eval import train_eval, evaluate

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Same preprocessing + model for every run (impute, scale, logistic regression)
def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

print("setup ready")

In [ ]:
results = {}

# Baseline: all 17 features
df = train_eval(
    make_pipeline(),
    train_df[FEATURES], y_train,
    val_df[FEATURES],   y_val,
    name="baseline (all 17)",
)
baseline_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
results["baseline"] = baseline_pr

print(f"Baseline validation PR-AUC (all 17 features): {baseline_pr:.4f}")
print(df.round(4))

In [ ]:
results = {"baseline": baseline_pr}

for group, cols in FEATURE_GROUPS.items():
    dropped_features = [f for f in FEATURES if f not in cols]

    df = train_eval(
        make_pipeline(),
        train_df[dropped_features], y_train,
        val_df[dropped_features],   y_val,
        name=f"without {group}",
    )
    val_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
    results[f"without {group}"] = val_pr
    print(f"without {group:12s} (dropped {len(cols)} features): PR-AUC = {val_pr:.4f}  drop = {baseline_pr - val_pr:+.4f}")

In [ ]:
import pandas as pd

rows = []
for name, pr in results.items():
    if name == "baseline":
        continue
    rows.append({
        "dropped group": name.replace("without ", ""),
        "PR-AUC": round(pr, 4),
        "drop vs baseline": round(baseline_pr - pr, 4),
    })

ablation_table = pd.DataFrame(rows).sort_values("drop vs baseline", ascending=False)
print(f"Baseline PR-AUC: {baseline_pr:.4f}")
print()
print(ablation_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sorted_table = ablation_table.sort_values("drop vs baseline")
values = sorted_table["drop vs baseline"]
labels = sorted_table["dropped group"]

bars = ax.barh(labels, values, color="tab:blue")

# Place labels with proper alignment based on sign
for bar, val in zip(bars, values):
    if val >= 0:
        ax.text(val + 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="left", fontsize=9)
    else:
        ax.text(val - 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="right", fontsize=9)

ax.axvline(0, color="black", linestyle="-", linewidth=0.8)

# Give some padding so labels don't touch the plot edges
x_min, x_max = values.min(), values.max()
padding = (x_max - x_min) * 0.15
ax.set_xlim(x_min - padding, x_max + padding)

ax.set_title("Feature-group ablation: validation PR-AUC drop when group is removed")
ax.set_xlabel("PR-AUC drop vs baseline (larger = more important)")
ax.set_ylabel("")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../reports/fig_p3_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.13 findings

Baseline validation PR-AUC (all 17 features): **0.7325**

| Dropped group | PR-AUC | Drop vs baseline |
|---|---|---|
| grid | 0.7183 | +0.0142 |
| qualifying | 0.7196 | +0.0129 |
| standings | 0.7209 | +0.0116 |
| circuit | 0.7275 | +0.0050 |
| form | 0.7303 | +0.0022 |
| context | 0.7355 | **-0.0030** |

**Interpretation**

*Ranking:* grid > qualifying > standings > circuit > form > context.

*All drops are small* (max 0.014). This is expected because features are 
correlated within and across groups: removing `grid` still leaves 
`qual_position` (~0.9 correlated), so the model compensates. The sum of the 
drops (0.043) is far smaller than the total signal, confirming that groups 
share information.

*Context is a negative drop* (-0.0030): removing `home_flag` slightly 
*improves* PR-AUC. This is within noise, but consistent with Q2 finding 
that home advantage disappears once you control for grid. `home_flag` is 
weak; keeping it is harmless, dropping it is also defensible.

**Takeaway.** Grid, qualifying, and standings are the core predictors. 
Circuit and form add modest value. Context (`home_flag`) adds nothing 
measurable — retained for completeness, flagged as weak.